---
# `FAISS`
---

**FAISS** stands for **Facebook AI Similarity Search**. It is an open-source library developed by Meta for **efficient similarity search and clustering of dense vectors**.

In GenAI, FAISS is commonly used to build:

* RAG applications
* Semantic search
* Document retrieval systems
* Recommendation systems
* Image similarity search
* Local vector-search prototypes

The most important idea is:

> **FAISS does not primarily act like a traditional database. It is a high-performance library for indexing vectors and finding vectors that are most similar to a query vector.**

---

# 1. Why Do We Need FAISS?

Suppose you have 1 million document chunks.

Each chunk is converted into an embedding:

```text
Document 1 → [0.21, 0.73, -0.12, ...]
Document 2 → [0.11, 0.65, -0.21, ...]
Document 3 → [0.92, -0.31, 0.44, ...]
...
Document 1,000,000 → [...]
```

Now a user asks:

> "What is deep learning?"

The question is converted into a vector:

```text
Question
   ↓
Embedding Model
   ↓
Query Vector
```

You need to find:

```text
Which document vectors
are closest to this query vector?
```

Naively comparing the query with every vector can become expensive.

FAISS provides optimized indexing and similarity-search algorithms to make this much faster.

---

# 2. Basic Architecture

Think of FAISS as:

```text
                  DOCUMENTS
                      │
                      ▼
               Embedding Model
                      │
                      ▼
                   Vectors
                      │
                      ▼
                    FAISS
                      │
              ┌───────┴───────┐
              ▼               ▼
           Index          Similarity
                          Search
```

At query time:

```text
User Question
      │
      ▼
Embedding Model
      │
      ▼
Query Vector
      │
      ▼
FAISS Index
      │
      ▼
Nearest Vectors
      │
      ▼
Document IDs
      │
      ▼
Relevant Documents
```

---

# 3. FAISS Is NOT an LLM

This distinction is critical.

FAISS:

```text
Vector
 ↓
Similarity Search
 ↓
Nearest Vectors
```

LLM:

```text
Prompt
 ↓
Language Model
 ↓
Generated Text
```

Therefore:

```text
FAISS ≠ GPT
FAISS ≠ Embedding Model
FAISS ≠ RAG
```

FAISS is one component that can be used **inside a RAG system**.

---

# 4. FAISS vs Embedding Model

These two are often confused.

### Embedding Model

Converts:

```text
Text → Vector
```

Example:

```text
"Machine Learning"
       ↓
Embedding Model
       ↓
[0.21, 0.72, -0.18, ...]
```

### FAISS

Takes vectors and searches them:

```text
Query Vector
     ↓
FAISS
     ↓
Most similar vectors
```

So:

```text
Embedding Model → creates vectors

FAISS → searches vectors
```

---

# 5. FAISS vs ChromaDB

You just learned ChromaDB, so this comparison is important.

| Feature              | FAISS                            | ChromaDB                          |
| -------------------- | -------------------------------- | --------------------------------- |
| Primary purpose      | Vector similarity search         | Vector database/vector store      |
| Vector indexing      | Excellent                        | Yes                               |
| Document storage     | Not its core purpose             | Yes                               |
| Metadata             | Not its core purpose             | Yes                               |
| Database features    | Limited                          | More database-like                |
| Persistence          | Possible, but you manage it      | Supported                         |
| Filtering            | Limited compared with vector DBs | Supported                         |
| Local development    | Excellent                        | Excellent                         |
| Distributed database | No                               | Not its primary design            |
| RAG                  | Excellent for retrieval layer    | Excellent for storage + retrieval |

The key distinction:

```text
FAISS
=
Vector Search Library
```

while:

```text
ChromaDB
=
Vector Store / Vector Database
```

FAISS can be used as the **vector search engine underneath an application**, while Chroma provides a more complete storage abstraction around vectors, documents, and metadata.

---

# 6. How FAISS Works

Suppose you have four vectors:

```text
A = [1, 2]
B = [2, 3]
C = [10, 10]
D = [11, 12]
```

Query:

```text
Q = [2, 2]
```

FAISS calculates a distance or similarity between the query and stored vectors.

Conceptually:

```text
Q → A : close
Q → B : very close
Q → C : far
Q → D : far
```

If you ask for:

```text
top_k = 2
```

FAISS returns:

```text
B
A
```

This is called **nearest-neighbor search**.

---

# 7. Similarity Search

The fundamental problem FAISS solves is:

> Given a query vector, find the closest vectors in a large collection.

This is generally called:

**Nearest Neighbor Search**

For example:

```text
Query Vector
     │
     ├── Vector 1 → distance 0.92
     ├── Vector 2 → distance 0.15
     ├── Vector 3 → distance 0.71
     ├── Vector 4 → distance 0.20
     └── Vector 5 → distance 1.30
```

The closest vectors are:

```text
Vector 2
Vector 4
```

---

# 8. Distance Metrics

FAISS supports different ways to measure similarity/distance.

The most important ones to understand are:

### L2 / Euclidean Distance

Measures geometric distance.

```text
A ●
   \
    \
     ● B
```

Smaller distance generally means more similar.

---

### Inner Product

Measures the dot product between vectors.

Higher values generally indicate greater alignment.

---

### Cosine Similarity

Cosine similarity measures the angle between vectors.

Conceptually:

```text
Same direction
     ↓
High similarity

Different direction
     ↓
Low similarity
```

Cosine similarity is especially common in semantic search.

A common implementation approach is to **normalize vectors and then use inner product**, because normalized inner product corresponds to cosine similarity.

---

# 9. Installing FAISS

For CPU environments:

```bash
pip install faiss-cpu
```

For GPU-enabled environments, FAISS also has GPU capabilities, although installation depends on the environment and package distribution.

For learning RAG locally, CPU FAISS is usually enough.

---

# 10. Basic FAISS Example

Let's create some vectors.

```python
import faiss
import numpy as np

vectors = np.array([
    [1, 2],
    [2, 3],
    [10, 10],
    [11, 12]
], dtype="float32")
```

We have:

```text
Vector 0 → [1, 2]
Vector 1 → [2, 3]
Vector 2 → [10, 10]
Vector 3 → [11, 12]
```

---

# 11. Create a FAISS Index

For simple Euclidean/L2 search:

```python
index = faiss.IndexFlatL2(2)
```

The `2` means:

> Each vector has 2 dimensions.

Then:

```python
index.add(vectors)
```

Now FAISS contains:

```text
Index
│
├── Vector 0
├── Vector 1
├── Vector 2
└── Vector 3
```

---

# 12. Search

Create a query:

```python
query = np.array([
    [2, 2]
], dtype="float32")
```

Search for the two nearest vectors:

```python
distances, indices = index.search(
    query,
    k=2
)
```

You get:

```text
distances
indices
```

`indices` tells you which vectors were the nearest neighbors.

For example:

```text
indices:
[[1, 0]]
```

means:

```text
Vector 1
Vector 0
```

were the two closest vectors.

---

# 13. Understanding `IndexFlatL2`

This is a very important FAISS index.

```python
index = faiss.IndexFlatL2(dimension)
```

It performs an exact search using L2 distance.

Suppose:

```text
1 million vectors
```

FAISS may compare the query against many/all stored vectors.

This is called:

**Brute-force / exact nearest-neighbor search.**

It provides high accuracy but can become expensive as the dataset grows.

---

# 14. Approximate Nearest Neighbor Search

For very large datasets, you often don't want to compare against every vector.

Instead, you can use:

**Approximate Nearest Neighbor (ANN)** methods.

The idea:

```text
Exact Search

Query
 ↓
Compare with every vector
 ↓
Find exact nearest neighbors
```

versus:

```text
Approximate Search

Query
 ↓
Search relevant regions
 ↓
Find very likely nearest neighbors
```

You trade some recall/accuracy for:

* Lower latency
* Less computation
* Better scalability

---

# 15. FAISS Index Types

FAISS provides multiple index structures.

You should understand these conceptually rather than memorizing every class.

### `IndexFlatL2`

```text
Exact search
+
L2 distance
```

Good for:

* Learning
* Small datasets
* Baselines

---

### `IndexFlatIP`

```text
Exact search
+
Inner Product
```

Useful when using inner-product similarity.

---

### IVF

**Inverted File Index**

Conceptually:

```text
All vectors
     ↓
Clusters
 ┌───┼───┐
 ▼   ▼   ▼
C1  C2  C3
```

At query time, FAISS searches only selected clusters instead of every vector.

This improves search efficiency.

---

### HNSW

**Hierarchical Navigable Small World**

Uses a graph-based structure for approximate nearest-neighbor search.

Conceptually:

```text
        A
       / \
      B---C
       \   \
        D---E
```

The graph allows efficient navigation toward nearby vectors.

---

### PQ

**Product Quantization**

Compresses vectors to reduce memory requirements and speed up search.

Conceptually:

```text
Large vector
     ↓
Split into parts
     ↓
Compress
     ↓
Smaller representation
```

---

# 16. FAISS in RAG

Now let's connect this to what you're learning.

Suppose:

```text
PDF
 ↓
Text Splitter
 ↓
100,000 chunks
```

Then:

```text
100,000 chunks
       ↓
Embedding Model
       ↓
100,000 vectors
       ↓
FAISS
```

At query time:

```text
"What is self-attention?"
        ↓
Embedding Model
        ↓
Query Vector
        ↓
FAISS
        ↓
Top 5 similar vectors
        ↓
Map IDs → Documents
        ↓
Prompt
        ↓
LLM
        ↓
Answer
```

This is a basic RAG architecture.

---

# 17. Important: FAISS Doesn't Know Your Documents

This is one of the biggest differences from a vector database.

Suppose FAISS returns:

```text
indices = [42, 17, 89]
```

FAISS fundamentally gives you vector/index results.

Your application needs to know:

```text
42 → chunk_42
17 → chunk_17
89 → chunk_89
```

So you may maintain a mapping:

```python
documents = {
    0: "Machine Learning is...",
    1: "Deep Learning uses...",
    2: "Transformers use...",
}
```

Then:

```python
documents[42]
```

retrieves the corresponding chunk.

This document/metadata management is something a more complete vector database or vector store abstraction can handle for you.

---

# 18. FAISS with LangChain

LangChain provides a FAISS vector-store integration.

Conceptually:

```text
LangChain
    │
    ▼
FAISS VectorStore
    │
    ▼
FAISS Index
```

Example:

```python
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

vector_store = FAISS.from_documents(
    documents,
    embeddings
)
```

Now LangChain handles the relationship between:

```text
Documents
+
Embeddings
+
FAISS index
```

---

# 19. Similarity Search with LangChain

```python
results = vector_store.similarity_search(
    "What is self-attention?",
    k=3
)
```

Now you get actual LangChain `Document` objects rather than just raw vector indexes.

For example:

```python
for doc in results:
    print(doc.page_content)
```

This is much easier for a RAG application.

---

# 20. FAISS + Retriever

You can also convert it into a retriever:

```python
retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)
```

Then:

```python
docs = retriever.invoke(
    "What is self-attention?"
)
```

Architecture:

```text
User
 │
 ▼
Question
 │
 ▼
Retriever
 │
 ▼
LangChain FAISS VectorStore
 │
 ▼
FAISS Index
 │
 ▼
Relevant Documents
```

---

# 21. Complete FAISS RAG Architecture

```text
                  DOCUMENT
                     │
                     ▼
              Document Loader
                     │
                     ▼
                Text Splitter
                     │
                     ▼
                  Chunks
                     │
                     ▼
              Embedding Model
                     │
                     ▼
                  Vectors
                     │
                     ▼
                 FAISS Index
                     │
                     │
─────────────────────┼─────────────────────
                     │
                  USER QUERY
                     │
                     ▼
              Embedding Model
                     │
                     ▼
                Query Vector
                     │
                     ▼
                  FAISS
                     │
                     ▼
              Similar Vectors
                     │
                     ▼
             Relevant Documents
                     │
                     ▼
                   Prompt
                     │
                     ▼
                  Chat LLM
                     │
                     ▼
                  Answer
```

---

# 22. FAISS vs ChromaDB vs Pinecone

This is a useful GenAI interview table.

| Feature            | FAISS                               | ChromaDB                 | Pinecone          |
| ------------------ | ----------------------------------- | ------------------------ | ----------------- |
| Type               | Search library                      | Vector DB/store          | Managed vector DB |
| Local              | Excellent                           | Excellent                | Primarily cloud   |
| Vector search      | Yes                                 | Yes                      | Yes               |
| Documents          | Via application/LangChain           | Yes                      | Yes               |
| Metadata           | Application-managed                 | Yes                      | Yes               |
| Persistence        | Possible                            | Yes                      | Managed           |
| Distributed        | No                                  | Not primary focus        | Yes               |
| Cloud managed      | No                                  | No/depends on deployment | Yes               |
| Learning           | Excellent                           | Excellent                | Good              |
| Production scaling | Requires surrounding infrastructure | Depends on deployment    | Strong            |

A simple rule:

```text
Learning / local prototype
        ↓
      FAISS

Easy local RAG
        ↓
     ChromaDB

Managed production vector infrastructure
        ↓
     Pinecone / Qdrant / etc.
```

The actual choice depends on workload, scale, operational requirements, latency, filtering needs, and deployment model.

---

# 23. FAISS Advantages

### 1. Very Fast

FAISS is highly optimized for vector search.

### 2. Efficient

It supports advanced indexing and compression techniques.

### 3. Open Source

You can use and customize it without depending on a managed service.

### 4. CPU and GPU Support

FAISS can take advantage of hardware acceleration depending on the index and environment.

### 5. Excellent for RAG Prototypes

Especially when you're building:

```text
PDF Chatbot
Document Search
Semantic Search
```

### 6. Many Index Types

You can choose between:

```text
Exact Search
ANN
Clustering
Quantization
Graph-based search
```

---

# 24. FAISS Limitations

### 1. Not a Complete Database

FAISS focuses on vector indexing/search.

You may need other systems for:

```text
Metadata
Authentication
Access control
Distributed serving
Application APIs
Operational management
```

### 2. Persistence Needs Planning

FAISS supports saving indexes, but application-level document and metadata persistence still needs to be designed.

### 3. No Built-in LLM

FAISS cannot generate answers.

### 4. No Embedding Model

FAISS doesn't decide how your text should be embedded.

### 5. Scaling Requires Architecture

For large production workloads, you may prefer a purpose-built vector database or a distributed retrieval architecture.

---

# 25. A Very Important Concept: Index

If you're preparing for AI/ML interviews, understand this term.

An **index** is a data structure that organizes vectors so similarity search can be performed efficiently.

Without an optimized index:

```text
Query
 ↓
Compare with every vector
 ↓
Expensive
```

With an appropriate index:

```text
Query
 ↓
Index
 ↓
Relevant search region
 ↓
Nearest neighbors
```

FAISS provides many different index structures optimized for different tradeoffs.

---

# 26. Exact vs Approximate Search

This is worth remembering.

### Exact

```text
Query
 ↓
Check all vectors
 ↓
Exact nearest neighbors
```

Advantages:

* High accuracy
* Simple

Disadvantages:

* Can be expensive at scale

---

### Approximate

```text
Query
 ↓
Index
 ↓
Search candidate vectors
 ↓
Approximate nearest neighbors
```

Advantages:

* Faster
* More scalable

Disadvantages:

* May miss some true nearest neighbors

This creates a classic tradeoff:

```text
Accuracy / Recall
       ↕
Search Speed
       ↕
Memory
```

---

# 27. Where FAISS Fits in Your GenAI Roadmap

You've already covered:

```text
Embeddings
      ↓
Vector Store
      ↓
ChromaDB
```

Now:

```text
FAISS
```

should be understood as the **vector indexing and similarity-search layer**.

Your progression should be:

```text
Tokenization
      ↓
Embeddings
      ↓
Vector Representation
      ↓
Similarity Search
      ↓
FAISS
      ↓
Vector Stores / Vector DBs
      ↓
Retrievers
      ↓
RAG
      ↓
Agents
```

---

# 28. Final Mental Model

Remember these three components:

```text
              TEXT
                │
                ▼
         EMBEDDING MODEL
                │
                ▼
             VECTOR
                │
                ▼
              FAISS
                │
        ┌───────┴────────┐
        ▼                ▼
    INDEXING        SIMILARITY
                       SEARCH
                         │
                         ▼
                 RELEVANT VECTORS
                         │
                         ▼
                    DOCUMENTS
                         │
                         ▼
                        LLM
                         │
                         ▼
                      ANSWER
```

### One-line definition

> **FAISS is a high-performance open-source library for indexing and similarity-searching dense vectors; in RAG, it is commonly used to retrieve the document chunks whose embeddings are most similar to a user's query.**

The key distinction from your previous topic is:

```text
Embedding Model → creates vectors

FAISS → efficiently searches vectors

ChromaDB → provides vector storage + retrieval capabilities

Retriever → exposes retrieval to the application

LLM → generates the final response
```

That separation is fundamental to understanding how modern **RAG systems** work.
